In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df = spark.read\
    .format("delta")\
        .load("/Volumes/workspace/default/my_volume/silver")


In [0]:
display(silver_df)

In [0]:
silver_df.printSchema()

In [0]:
from pyspark.sql.functions import round

In [0]:
gold_outlet_sales = (
    silver_df.groupBy("outlet_id")\
        .agg(
            sum("item_outlet_sales").alias("total_sales"),
            avg("item_outlet_sales").alias("total_average_sales"),
            count("item_outlet_sales").alias("Total_count_sales")

        )
        
)

In [0]:
display(gold_outlet_sales)

# upto this moment we have created the bussiness presentable data using the cleaned datsa from the silver layer. 

In [0]:
gold_outlet_sales.printSchema()

In [0]:
gold_outlet_sales.count()

# here we will chcek for the null values in this particular column

In [0]:
from pyspark.sql.functions import (
    round,
    col,
    column

)

In [0]:
for column in gold_outlet_sales.columns:
    null_count = gold_outlet_sales.filter(
        col(column).isNull()
    ).count()

    print(f"{null_count} : {column} Null Values")

### saving the gold_outlet_sales table as the delta format

In [0]:
gold_outlet_sales.write\
    .format("delta")\
        .mode("overwrite")\
            .save("/Volumes/workspace/default/my_volume/gold_outlet_sales")

In [0]:
gold_outlet_sales_check = spark.read\
    .format("delta")\
        .load("/Volumes/workspace/default/my_volume/gold_outlet_sales")

In [0]:
display(gold_outlet_sales)

In [0]:
gold_outlet_sales.printSchema()

### now here we will create gold product performance

In [0]:
gold_item_sales = (
    silver_df
    .groupBy("item_id")
    .agg(
        round(sum("item_outlet_sales"), 2).alias("total_sales"),
        round(avg("item_outlet_sales"), 2).alias("average_sales"),
        countDistinct("outlet_id").alias("outlets_sold"),
        count("*").alias("total_records")
    )
)

In [0]:
display(
    gold_item_sales.orderBy(
        col("total_sales").desc()
    )
)

In [0]:
gold_item_sales.write\
    .format("delta")\
        .mode("overwrite")\
            .save("/Volumes/workspace/default/my_volume/gold_item_sales")

In [0]:
gold_item_sales = spark.read\
    .format("delta")\
        .load("/Volumes/workspace/default/my_volume/gold_item_sales")

In [0]:
gold_item_sales.show(truncate = False)

In [0]:
gold_outlet_type_sales = (
    silver_df
    .groupBy("outlet_type")
    .agg(
        round(sum("item_outlet_sales"), 2).alias("total_sales"),
        round(avg("item_outlet_sales"), 2).alias("average_sales"),
        countDistinct("outlet_id").alias("number_of_outlets"),
        count("*").alias("total_records")
    )
)

## we are using orderBy in order to sort it and also we are sorting it on the dbasis of total_sales in the descending order so this is how it is wroking.

In [0]:
display(
    gold_outlet_type_sales.orderBy(
        col("total_sales").desc()
    )

)

In [0]:
gold_outlet_type_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/my_volume/gold_outlet_type_sales")

In [0]:
gold_outlet_type_sales_check = spark.read \
    .format("delta") \
    .load("/Volumes/workspace/default/my_volume/gold_outlet_type_sales")

In [0]:
display(gold_outlet_type_sales)

In [0]:
gold_outlet_type_sales.printSchema()

In [0]:
gold_outlet_type_sales.count()

In [0]:
print("===== GOLD LAYER VALIDATION =====")

print("Outlet Sales records:",
      gold_outlet_sales.count())

print("Item Sales records:",
      gold_item_sales.count())

print("Outlet Type Sales records:",
      gold_outlet_type_sales.count())